<a href="https://colab.research.google.com/github/meghansn/mental-health-llm-pipeline/blob/rag-disorders/Symptom_Table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-cloud-bigquery pandas

In [ ]:
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

client = bigquery.Client(project="mental-health-llm-pip")

schema = [
    bigquery.SchemaField("disorder_id", "STRING"),
    bigquery.SchemaField("disorder_name", "STRING"),
    bigquery.SchemaField("icd10_code", "STRING"),
    bigquery.SchemaField("symptoms", "STRING"),
]

dataset_id = "mental-health-llm-pip.mental_health"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"
dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Created dataset {dataset.project}.{dataset.dataset_id}")

table_id = "mental-health-llm-pip.mental_health.dim_disorders"

table = bigquery.Table(table_id, schema=schema)

table = client.create_table(table, exists_ok=True)

print(f"Created {table_id}")

In [ ]:
table = client.get_table(table_id)

print(f"Table: {table.table_id}")
print(f"Rows: {table.num_rows}")

for field in table.schema:
    print(field.name, field.field_type)

In [ ]:
bigquery.SchemaField("description", "STRING"),

In [ ]:
schema = [
    bigquery.SchemaField("disorder_id", "STRING"),
    bigquery.SchemaField("disorder_name", "STRING"),
    bigquery.SchemaField("dsm5_category", "STRING"),
    bigquery.SchemaField("icd10_code", "STRING"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("diagnostic_criteria", "STRING"),
    bigquery.SchemaField("symptoms", "STRING"),
]

In [ ]:
import pandas as pd

data = [
    {
        "disorder_id": "D001",
        "disorder_name": "Major Depressive Disorder",
        "dsm5_category": "Depressive Disorders",
        "icd10_code": "F33.1",
        "description": "Mood disorder characterized by persistent sadness and loss of interest.",
        "diagnostic_criteria": "Persistent low mood, loss of interest, fatigue, sleep disturbance, concentration difficulties.",
        "symptoms": "depressed mood; anhedonia; fatigue; insomnia; hopelessness"
    },
    {
        "disorder_id": "D002",
        "disorder_name": "Generalized Anxiety Disorder",
        "dsm5_category": "Anxiety Disorders",
        "icd10_code": "F41.1",
        "description": "Excessive and persistent worry that is difficult to control.",
        "diagnostic_criteria": "Excessive worry, restlessness, irritability, fatigue, concentration difficulties.",
        "symptoms": "worry; restlessness; irritability; fatigue"
    }
]

df = pd.DataFrame(data)

In [ ]:
# Update the table schema to include all fields from the DataFrame
table_ref = client.get_table(table_id) # Get current table reference
table_ref.schema = schema # 'schema' variable was defined in a previous cell with the expanded schema
client.update_table(table_ref, ["schema"]) # Apply the schema update

job = client.load_table_from_dataframe(
    df,
    table_id
)

job.result()

print("Loaded rows")

In [ ]:
query = """
SELECT *
FROM `mental-health-llm-pip.mental_health.dim_disorders`
LIMIT 10
"""

df = client.query(query).to_dataframe()

df.head()

In [ ]:
query = """INSERT INTO `mental-health-llm-pip.mental_health.dim_disorders`
(
  disorder_id,
  disorder_name,
  dsm5_category,
  icd10_code,
  description,
  diagnostic_criteria,
  symptoms
)
VALUES

(
  'D001',
  'Major Depressive Disorder',
  'Depressive Disorders',
  'F33.1',
  'Persistent low mood and reduced interest affecting daily functioning.',
  'Low mood, loss of interest, fatigue, sleep changes, concentration difficulties.',
  'depressed mood;anhedonia;fatigue;insomnia;hopelessness'
),

(
  'D002',
  'Persistent Depressive Disorder',
  'Depressive Disorders',
  'F34.1',
  'Chronic depressive symptoms lasting for an extended period.',
  'Long-term low mood, low energy, poor self-esteem, hopelessness.',
  'low mood;fatigue;hopelessness;low self-esteem'
),

(
  'D003',
  'Generalized Anxiety Disorder',
  'Anxiety Disorders',
  'F41.1',
  'Excessive and difficult-to-control worry across multiple situations.',
  'Persistent worry, restlessness, fatigue, irritability, concentration problems.',
  'worry;restlessness;irritability;fatigue;concentration difficulties'
),

(
  'D004',
  'Panic Disorder',
  'Anxiety Disorders',
  'F41.0',
  'Recurrent panic attacks accompanied by concern about future attacks.',
  'Unexpected panic attacks and fear of recurrence.',
  'panic attacks;heart palpitations;fear;shortness of breath'
),

(
  'D005',
  'Social Anxiety Disorder',
  'Anxiety Disorders',
  'F40.10',
  'Intense fear of social situations and negative evaluation.',
  'Avoidance of social situations and excessive fear of embarrassment.',
  'social fear;avoidance;self-consciousness;anxiety'
),

(
  'D006',
  'Bipolar I Disorder',
  'Bipolar and Related Disorders',
  'F31.9',
  'Condition involving episodes of elevated mood and increased activity.',
  'Manic episodes with significant mood and behavioral changes.',
  'mania;elevated mood;decreased need for sleep;impulsivity'
),

(
  'D007',
  'Bipolar II Disorder',
  'Bipolar and Related Disorders',
  'F31.81',
  'Condition involving hypomanic and depressive episodes.',
  'Hypomania combined with periods of depression.',
  'hypomania;depression;mood swings;fatigue'
),

(
  'D008',
  'Post-Traumatic Stress Disorder',
  'Trauma- and Stressor-Related Disorders',
  'F43.10',
  'Condition that may occur after exposure to a traumatic event.',
  'Intrusive memories, avoidance, hyperarousal, emotional distress.',
  'flashbacks;nightmares;avoidance;hypervigilance'
),

(
  'D009',
  'Obsessive-Compulsive Disorder',
  'Obsessive-Compulsive and Related Disorders',
  'F42',
  'Condition characterized by obsessions and repetitive behaviors.',
  'Persistent intrusive thoughts and compulsive actions.',
  'obsessions;compulsions;repetitive behaviors;intrusive thoughts'
),

(
  'D010',
  'Adjustment Disorder',
  'Trauma- and Stressor-Related Disorders',
  'F43.20',
  'Emotional or behavioral symptoms occurring in response to a stressor.',
  'Distress following identifiable life changes or stressors.',
  'stress;anxiety;sadness;difficulty coping'
);"""

query_job = client.query(query) # API request
query_job.result() # Wait for the query to complete

table_id = "mental-health-llm-pip.mental_health.dim_disorders" # Define table_id here
print(f"Inserted {query_job.num_dml_affected_rows} rows into {table_id}")

In [ ]:
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

client = bigquery.Client(project="mental-health-llm-pip")

query = """ALTER TABLE `mental-health-llm-pip.mental_health.dim_disorders`
ADD COLUMN retrieval_text STRING;"""

query_job = client.query(query) # API request
query_job.result() # Wait for the query to complete

In [ ]:
query = """UPDATE `mental-health-llm-pip.mental_health.dim_disorders`

SET retrieval_text =

CONCAT(

  disorder_name,

  CHR(10), CHR(10), 'DSM Category: ', dsm5_category,

  CHR(10), 'ICD10 Code: ', icd10_code,

  CHR(10), CHR(10), 'Description: ', description,

  CHR(10), CHR(10), 'Diagnostic Criteria: ', diagnostic_criteria,

  CHR(10), CHR(10), 'Symptoms: ', symptoms

)

WHERE TRUE;"""

query_job = client.query(query) # API request
query_job.result() # Wait for the query to complete

In [ ]:
query = """SELECT
  disorder_name,
  retrieval_text
FROM `mental-health-llm-pip.mental_health.dim_disorders`
LIMIT 3;"""

query_job = client.query(query) # API request

# Convert to DataFrame and display
df_result = query_job.to_dataframe()
display(df_result.head())

In [ ]:
query = """SELECT

  disorder_name,

  LENGTH(retrieval_text) AS text_length

FROM `mental-health-llm-pip.mental_health.dim_disorders`

ORDER BY disorder_name;"""

query_job = client.query(query) # API request

# Convert to DataFrame and display
df_result = query_job.to_dataframe()
display(df_result.head())

In [ ]:
!pip install -q google-cloud-aiplatform vertexai

In [ ]:
import pandas as pd
import vertexai

from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel

In [ ]:
PROJECT_ID = "mental-health-llm-pip"
LOCATION = "us-central1"

client = bigquery.Client(project=PROJECT_ID)

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

In [ ]:
query = """
SELECT
    disorder_id,
    disorder_name,
    retrieval_text
FROM `mental-health-llm-pip.mental_health.dim_disorders`
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head()

In [ ]:
model = TextEmbeddingModel.from_pretrained(
    "text-embedding-004"
)

print("Model loaded")

In [ ]:
sample_text = df.iloc[0]["retrieval_text"]

embedding = model.get_embeddings(
    [sample_text]
)[0].values

print(type(embedding))
print(len(embedding))

In [ ]:
embeddings = []

for _, row in df.iterrows():

    vector = model.get_embeddings(
        [row["retrieval_text"]]
    )[0].values

    embeddings.append({
        "disorder_id": row["disorder_id"],
        "disorder_name": row["disorder_name"],
        "retrieval_text": row["retrieval_text"],
        "embedding": vector
    })

print(f"Generated {len(embeddings)} embeddings")

In [ ]:
embedding_df = pd.DataFrame(embeddings)

embedding_df.head()

In [ ]:
embedding_df.info()

In [ ]:
table_id = "mental-health-llm-pip.mental_health.disorder_embeddings"

job = client.load_table_from_dataframe(
    embedding_df,
    table_id
)

job.result()

print("Embeddings uploaded")

In [ ]:
patient_symptoms = """
intrusive thoughts
repetitive behaviors
checking rituals
compulsions
"""

In [ ]:
query_embedding = model.get_embeddings(
    [patient_symptoms]
)[0].values

print(len(query_embedding))

In [ ]:
query = """
SELECT
    disorder_name,
    embedding
FROM `mental-health-llm-pip.mental_health.disorder_embeddings`
"""

disorders_df = client.query(query).to_dataframe()

print(disorders_df.shape)

In [ ]:
!pip install -q scikit-learn

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

disorders_df["similarity"] = disorders_df["embedding"].apply(
    lambda x: cosine_similarity(
        [query_embedding],
        [x]
    )[0][0]
)

In [ ]:
results = disorders_df.sort_values(
    "similarity",
    ascending=False
)

results[
    ["disorder_name", "similarity"]
].head(5)